# Pipeline — crypto_to_silver

Orchestrates the full crypto data pipeline from API fetch through to the Silver layer.
Each stage is individually toggleable — skip stages whose data is already up to date.

| Stage | Flag | Notebook |
|---|---|---|
| Landing | `RUN_LANDING` | `02_fetch_crypto_for_date_binance.ipynb` |
| Bronze  | `RUN_BRONZE`  | `01_crypto_landing_to_bronze.ipynb` |
| Silver  | `RUN_SILVER`  | `01_crypto_bronze_to_silver.ipynb` |

**Execution:** stages run sequentially; on first failure the pipeline stops immediately.
All log events are written to `s3a://warehouse/logs/pipeline_runs` in a single bulk write at the end.

In [ ]:
PIPELINE_NAME = "crypto_to_silver"

# Toggle stages — set to False to skip
RUN_LANDING = True    # fetch fresh data from Binance API → landing zone
RUN_BRONZE  = True    # landing CSVs → bronze Delta table
RUN_SILVER  = False   # bronze → silver Delta table (not yet implemented)

# Shared
LOG_PATH = "s3a://warehouse/logs/pipeline_runs"

# Landing parameters
WATCHLIST = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT"]
DAYS      = 365

# Bronze parameters
SYMBOLS     = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT"]
LANDING_DIR = "/home/landing/crypto/daily"
BRONZE_PATH = "s3a://warehouse/bronze/crypto"

# Silver parameters
SILVER_PATH = "s3a://warehouse/silver/crypto"

## Setup

In [ ]:
import os
import uuid
import papermill as pm
from datetime import datetime, timezone
from pyspark.sql import SparkSession, Row
from delta import configure_spark_with_delta_pip

MINIO_ENDPOINT   = os.environ["MINIO_ENDPOINT"]
MINIO_ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]
MINIO_SECRET_KEY = os.environ["MINIO_SECRET_KEY"]

builder = (
    SparkSession.builder
    .appName("pipeline-logger")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

run_id = str(uuid.uuid4())
print(f"Run ID      : {run_id}")
print(f"Pipeline    : {PIPELINE_NAME}")
print(f"Started at  : {datetime.now(timezone.utc).isoformat()}")
print(f"Stages      : landing={RUN_LANDING}  bronze={RUN_BRONZE}  silver={RUN_SILVER}")

## Pipeline Stages

In [ ]:
steps = []

if RUN_LANDING:
    steps.append({
        "name":       "fetch_crypto",
        "notebook":   "/home/notebooks/landing/02_fetch_crypto_for_date_binance.ipynb",
        "parameters": {"WATCHLIST": WATCHLIST, "DAYS": DAYS},
    })

if RUN_BRONZE:
    steps.append({
        "name":       "bronze_crypto",
        "notebook":   "/home/notebooks/bronze/01_crypto_landing_to_bronze.ipynb",
        "parameters": {"SYMBOLS": SYMBOLS, "LANDING_DIR": LANDING_DIR, "BRONZE_PATH": BRONZE_PATH},
    })

if RUN_SILVER:
    steps.append({
        "name":       "silver_crypto",
        "notebook":   "/home/notebooks/silver/01_crypto_bronze_to_silver.ipynb",
        "parameters": {"BRONZE_PATH": BRONZE_PATH, "SILVER_PATH": SILVER_PATH},
    })

print(f"{len(steps)} stage(s) queued: {[s['name'] for s in steps]}")

## Execute

In [ ]:
log_events  = []
pipeline_ok = True

for step in steps:
    started_at    = datetime.now(timezone.utc)
    status        = "success"
    error_message = None

    print(f"\n[{step['name']}] Starting...")
    try:
        pm.execute_notebook(
            step["notebook"],
            f"/tmp/{step['name']}_output.ipynb",
            parameters=step["parameters"],
            kernel_name="python3",
        )
        print(f"[{step['name']}] Done.")
    except Exception as exc:
        status        = "failed"
        error_message = str(exc)
        pipeline_ok   = False
        print(f"[{step['name']}] FAILED: {exc}")

    finished_at = datetime.now(timezone.utc)
    log_events.append({
        "run_id":           run_id,
        "pipeline_name":    PIPELINE_NAME,
        "step_name":        step["name"],
        "notebook_path":    step["notebook"],
        "status":           status,
        "started_at":       started_at,
        "finished_at":      finished_at,
        "duration_seconds": (finished_at - started_at).total_seconds(),
        "error_message":    error_message,
    })

    if not pipeline_ok:
        print("\nPipeline stopped due to failure.")
        break

print(f"\nPipeline {'completed successfully' if pipeline_ok else 'FAILED'}.")

## Write Logs

In [ ]:
df_logs = spark.createDataFrame([Row(**e) for e in log_events])
df_logs.write.format("delta").mode("append").save(LOG_PATH)

print(f"Logged {len(log_events)} event(s) to {LOG_PATH}\n")
df_logs.select("step_name", "status", "duration_seconds", "error_message").show(truncate=False)